In [1]:
from pyspark.sql import SparkSession
spark= SparkSession.builder\
       .appName("ETL_pipelines")\
       .getOrCreate()
print("spark session ready")

spark session ready


In [ ]:
import dlt

@dlt.table(
    name="bronze_energy_logs"
)
def bronze_energy_logs():

    logs = [
        (1,2,1,"2026-05-01 09:00:00",1.2000),
        (2,2,1,"2026-05-01 13:00:00",1.1000),
        (3,2,1,"2026-05-01 18:00:00",1.3500),
        (4,1,1,"2026-05-01 20:00:00",0.1500),
        (5,1,1,"2026-05-02 21:00:00",0.1800),
        (6,3,2,"2026-05-01 10:00:00",0.0500),
        (7,3,2,"2026-05-01 15:00:00",0.0600),
        (8,3,2,"2026-05-02 11:00:00",0.0550),
        (9,5,3,"2026-05-01 08:00:00",0.4000),
        (10,5,3,"2026-05-01 12:00:00",0.4200),
        (11,5,3,"2026-05-02 08:00:00",0.4100),
        (12,5,3,"2026-05-02 20:00:00",0.4500),
        (13,7,4,"2026-05-01 09:30:00",0.0800),
        (14,7,4,"2026-05-01 17:30:00",0.0900),
        (15,7,4,"2026-05-02 09:30:00",0.0850),
        (16,8,4,"2026-05-01 00:00:00",0.0200),
        (17,8,4,"2026-05-01 06:00:00",0.0220),
        (18,8,4,"2026-05-02 00:00:00",0.0210),
        (19,2,1,"2026-06-01 10:00:00",1.5000),
        (20,2,1,"2026-06-01 15:00:00",1.4500),
        (21,5,3,"2026-06-06 08:00:00",0.4800),
        (22,5,3,"2026-06-06 20:00:00",0.5000),
        (23,2,1,"2026-07-01 12:00:00",2.5000),
        (24,5,3,"2026-07-01 08:00:00",0.6000),
        (25,3,2,"2026-07-02 09:00:00",0.0700),
        (26,3,2,"2026-07-02 14:00:00",0.0750),
        (27,7,4,"2026-07-03 10:00:00",0.1000),
        (28,7,4,"2026-07-03 18:00:00",0.1100),
        (29,8,4,"2026-07-04 00:00:00",0.0300),
        (30,8,4,"2026-07-04 06:00:00",0.0320)
    ]

    columns = ["log_id","device_id","room_id","timestamp","energy_kwh"]
    df = spark.createDataFrame(logs, columns)

    return df

In [ ]:
from pyspark.sql.functions import col, to_timestamp

@dlt.table(
    name="silver_energy_logs"
)
def silver_energy_logs():

    df = dlt.read("bronze_energy_logs")

    df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))
    df = df.filter(col("timestamp").isNotNull())

    df = df.withColumn("energy_kwh", col("energy_kwh").cast("double"))

    return df

In [ ]:
from pyspark.sql.functions import sum, to_date

@dlt.table(
    name="gold_daily_energy_summary"
)
def gold_daily_energy_summary():

    df = dlt.read("silver_energy_logs")

    df = df.withColumn("log_date", to_date(col("timestamp")))

    daily_summary = df.groupBy("room_id", "log_date") \
        .agg(sum("energy_kwh").alias("daily_total_energy_kwh"))

    return daily_summary

In [ ]:
from pyspark.sql.functions import weekofyear, year

@dlt.table(
    name="gold_weekly_energy_summary"
)
def gold_weekly_energy_summary():

    df = dlt.read("silver_energy_logs")

    weekly_summary = df.groupBy(
        "room_id",
        year(col("timestamp")).alias("year"),
        weekofyear(col("timestamp")).alias("week_no")
    ).agg(
        sum("energy_kwh").alias("weekly_total_energy_kwh")
    )

    return weekly_summary

In [ ]:
from pyspark.sql.functions import sum

@dlt.table(
    name="gold_device_energy_summary"
)
def gold_device_energy_summary():

    df = dlt.read("silver_energy_logs")

    device_summary = df.groupBy("device_id", "room_id") \
        .agg(sum("energy_kwh").alias("total_energy_kwh"))

    return device_summary

In [ ]:
from pyspark.sql.functions import when

@dlt.table(
    name="gold_daily_wastage_alerts"
)
def gold_daily_wastage_alerts():

    df = dlt.read("silver_energy_logs")
    df = df.withColumn("log_date", to_date(col("timestamp")))

    daily_device_usage = df.groupBy("device_id", "room_id", "log_date") \
        .agg(sum("energy_kwh").alias("daily_energy_kwh"))

    daily_device_usage = daily_device_usage.withColumn(
        "alert",
        when(col("daily_energy_kwh") > 5, "HIGH_USAGE_ALERT").otherwise("NORMAL")
    )

    return daily_device_usage